In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")


#typed column builders: each returns (raw Bronze column, typed expression)
def text(raw):
    return raw, F.trim(F.col(raw))


def code(raw):
    return raw, F.upper(F.trim(F.col(raw)))


def integer(raw):
    return raw, F.trim(F.col(raw)).try_cast("int")


def us_date(raw):
    return raw, F.to_date(F.try_to_timestamp(F.trim(F.col(raw)), F.lit("M/d/yyyy")))


def usd(raw):
    return raw, F.regexp_replace(F.col(raw), r"[$,\s]", "").try_cast("decimal(18,2)")


def rate(raw):
    return raw, F.trim(F.col(raw)).try_cast("decimal(18,6)")


def derived(expression):
    return None, expression


def write_silver(df, table_name):
    (
        df.withColumn("_silver_processed_at_utc", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

In [ ]:
REFERENCE_TABLES = {
    "silver.customers": {
        "source": "bronze.customers_raw",
        "key_columns": ["customer_key"],
        "columns": {
            "customer_key": integer("customer_key"),
            "gender": text("gender_raw"),
            "customer_name": text("customer_name"),
            "city": text("city"),
            "state_code": text("state_code_raw"),
            "state": text("state"),
            "zip_code": text("zip_code_raw"),
            "country": text("country"),
            "continent": text("continent"),
            "birth_date": us_date("birthday_raw"),
        },
    },
    "silver.products": {
        "source": "bronze.products_raw",
        "key_columns": ["product_key"],
        "columns": {
            "product_key": integer("product_key"),
            "product_name": text("product_name"),
            "brand": text("brand"),
            "color": text("color"),
            "unit_cost_usd": usd("unit_cost_usd_raw"),
            "unit_price_usd": usd("unit_price_usd_raw"),
            "subcategory_key": integer("subcategory_key"),
            "subcategory": text("subcategory"),
            "category_key": integer("category_key"),
            "category": text("category"),
        },
    },
    "silver.stores": {
        "source": "bronze.stores_raw",
        "key_columns": ["store_key"],
        "columns": {
            "store_key": integer("store_key"),
            "country": text("country"),
            "state": text("state"),
            "square_meters": integer("square_meters_raw"),
            "open_date": us_date("open_date_raw"),
            "is_online": derived(F.trim(F.col("store_key")).try_cast("int") == 0),
        },
    },
    "silver.exchange_rates": {
        "source": "bronze.exchange_rates_raw",
        "key_columns": ["rate_date", "currency_code"],
        "columns": {
            "rate_date": us_date("rate_date_raw"),
            "currency_code": code("currency_code_raw"),
            "rate_per_usd": rate("exchange_rate_raw"),
        },
    },
}

In [ ]:
def build_reference(table_name, spec):
    columns = spec["columns"]
    raw_columns = {name: raw for name, (raw, _) in columns.items() if raw}

    bronze_df = spark.table(spec["source"])
    checked_df = bronze_df.select(
        *[expression.alias(name) for name, (_, expression) in columns.items()],
        *[F.col(raw).alias(f"__raw_{name}") for name, raw in raw_columns.items()],
    )

    #a value that was present in Bronze but could not be typed is a parse failure
    parse_failures = checked_df.agg(*[
        F.sum(
            F.when(F.col(f"__raw_{name}").isNotNull() & F.col(name).isNull(), 1).otherwise(0)
        ).alias(name)
        for name in raw_columns
    ]).first().asDict()

    silver_df = checked_df.select(*columns)
    write_silver(silver_df, table_name)

    target_df = spark.table(table_name)
    key_columns = spec["key_columns"]
    return {
        "table": table_name,
        "bronze_rows": bronze_df.count(),
        "silver_rows": target_df.count(),
        "duplicate_key_groups": (
            target_df.groupBy(*key_columns).count().filter(F.col("count") > 1).count()
        ),
        "null_key_rows": target_df.filter(
            " OR ".join(f"{c} IS NULL" for c in key_columns)
        ).count(),
        "parse_failures": {name: n for name, n in parse_failures.items() if n},
    }


reference_results = [build_reference(name, spec) for name, spec in REFERENCE_TABLES.items()]
for result in reference_results:
    print(result)

In [ ]:
#daily rates: every date and currency gets the latest rate on or before that date, never a later one
rates_df = spark.table("silver.exchange_rates")
last_order_date = (
    spark.table("bronze.orders_raw")
    .select(F.to_date(F.try_to_timestamp(F.trim("order_date_raw"), F.lit("M/d/yyyy"))).alias("d"))
    .agg(F.max("d"))
    .first()[0]
)
first_rate_date, last_rate_date = rates_df.agg(F.min("rate_date"), F.max("rate_date")).first()
calendar_end = max(last_rate_date, last_order_date)

calendar_df = spark.sql(
    f"SELECT explode(sequence(DATE'{first_rate_date}', DATE'{calendar_end}')) AS rate_date"
)
carry_forward = (
    Window.partitionBy("currency_code")
    .orderBy("rate_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

daily_df = (
    calendar_df
    .crossJoin(rates_df.select("currency_code").distinct())
    .join(rates_df.select("rate_date", "currency_code", "rate_per_usd"), ["rate_date", "currency_code"], "left")
    .withColumn("source_rate_date", F.when(F.col("rate_per_usd").isNotNull(), F.col("rate_date")))
    .select(
        "rate_date",
        "currency_code",
        F.last("rate_per_usd", ignorenulls=True).over(carry_forward).alias("rate_per_usd"),
        F.last("source_rate_date", ignorenulls=True).over(carry_forward).alias("source_rate_date"),
    )
    .filter(F.col("rate_per_usd").isNotNull())
    .withColumn("is_carried_forward", F.col("source_rate_date") != F.col("rate_date"))
)

write_silver(daily_df, "silver.exchange_rates_daily")

daily = spark.table("silver.exchange_rates_daily")
print(f"Daily rates: {daily.count():,} rows from {first_rate_date} to {calendar_end}")
print(f"Carried-forward rows: {daily.filter('is_carried_forward').count():,}")
print(f"Rows using a future rate: {daily.filter('source_rate_date > rate_date').count():,}")

In [ ]:
#validate
for result in reference_results:
    table = result["table"]
    assert result["silver_rows"] == result["bronze_rows"], f"{table}: row count mismatch"
    assert result["duplicate_key_groups"] == 0, f"{table}: duplicate keys"
    assert result["null_key_rows"] == 0, f"{table}: null keys"
    assert not result["parse_failures"], f"{table}: parse failures {result['parse_failures']}"

assert daily.filter("source_rate_date > rate_date").count() == 0
assert daily.groupBy("rate_date", "currency_code").count().filter("count > 1").count() == 0

display(spark.createDataFrame([
    {k: v for k, v in r.items() if k != "parse_failures"} for r in reference_results
]))